In [2]:
%pip install kaggle

Note: you may need to restart the kernel to use updated packages.


In [3]:
%pip install kagglehub

Note: you may need to restart the kernel to use updated packages.


In [4]:
#Import some libraries
import pandas as pd
import numpy as np

#Import zipfile library (Use to extract zipfile from kaggle)
import zipfile


### Data gathering and exploration

In [6]:
import kagglehub

# Download latest version of london_bike dataset from kaggle
path = kagglehub.dataset_download("hmavrodiev/london-bike-sharing-dataset")

# Show file path 
print("Path to dataset files:", path)

#Check file from kagglehub path
test_bike = pd.read_csv("london_merged.csv")

Path to dataset files: C:\Users\buidu\.cache\kagglehub\datasets\hmavrodiev\london-bike-sharing-dataset\versions\1


In [7]:
#read csv file as a pandas dataframe
bike = pd.read_csv("london_merged.csv")

In [8]:
#explore data
bike.info()

<class 'pandas.DataFrame'>
RangeIndex: 17414 entries, 0 to 17413
Data columns (total 10 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   timestamp     17414 non-null  str    
 1   cnt           17414 non-null  int64  
 2   t1            17414 non-null  float64
 3   t2            17414 non-null  float64
 4   hum           17414 non-null  float64
 5   wind_speed    17414 non-null  float64
 6   weather_code  17414 non-null  float64
 7   is_holiday    17414 non-null  float64
 8   is_weekend    17414 non-null  float64
 9   season        17414 non-null  float64
dtypes: float64(8), int64(1), str(1)
memory usage: 1.3 MB


In [9]:
bike.shape

(17414, 10)

In [10]:
bike

,timestamp,cnt,t1,t2,hum,wind_speed,weather_code,is_holiday,is_weekend,season
0,2015-01-04 00:00:00,182,3.0,2.0,93.0,6.0,3.0,0.0,1.0,3.0
1,2015-01-04 01:00:00,138,3.0,2.5,93.0,5.0,1.0,0.0,1.0,3.0
2,2015-01-04 02:00:00,134,2.5,2.5,96.5,0.0,1.0,0.0,1.0,3.0
3,2015-01-04 03:00:00,72,2.0,2.0,100.0,0.0,1.0,0.0,1.0,3.0
4,2015-01-04 04:00:00,47,2.0,0.0,93.0,6.5,1.0,0.0,1.0,3.0
...,...,...,...,...,...,...,...,...,...,...
17409,2017-01-03 19:00:00,1042,5.0,1.0,81.0,19.0,3.0,0.0,0.0,3.0
17410,2017-01-03 20:00:00,541,5.0,1.0,81.0,21.0,4.0,0.0,0.0,3.0
17411,2017-01-03 21:00:00,337,5.5,1.5,78.5,24.0,4.0,0.0,0.0,3.0
17412,2017-01-03 22:00:00,224,5.5,1.5,76.0,23.0,4.0,0.0,0.0,3.0


In [11]:
# there are 10 columns in this dataset --> check the unique value from weather_code and season columns
bike.weather_code.value_counts()

weather_code
1.0     6150
2.0     4034
3.0     3551
7.0     2141
4.0     1464
26.0      60
10.0      14
Name: count, dtype: int64

In [12]:
bike.season.value_counts()

season
0.0    4394
1.0    4387
3.0    4330
2.0    4303
Name: count, dtype: int64

### Metadata:

"timestamp" - timestamp field for grouping the data

"cnt" - the count of a new bike shares

"t1" - real temperature in C

"t2" - temperature in C "feels like"

"hum" - humidity in percentage

"wind_speed" - wind speed in km/h

"weather_code" - category of the weather

"is_holiday" - boolean field - 1 holiday / 0 non holiday

"is_weekend" - boolean field - 1 if the day is weekend

"season" - category field meteorological seasons: 0-spring ; 1-summer; 2-fall; 3-winter.


"weathe_code" category description:

1 = Clear ; mostly clear but have some values with haze/fog/patches of fog/ fog in vicinity 2 = scattered clouds / few clouds 3 = Broken clouds 4 = Cloudy 7 = Rain/ light Rain shower/ Light rain 10 = rain with thunderstorm 26 = snowfall 94 = Freezing Fog

### Data manipulation

In [13]:
# specify the column names in dictionary
new_cols_dict = {
    'timestamp' : 'time',
    'cnt' : 'count of a new bike shares',
    't1' : 'real temperature in C',
    't2' : 'temperature in C feels like',
    'hum' : 'humidity_percent',
    'wind_speed' : 'wind_speed_km/h',
    'weather_code' : 'weather',
    'is_holiday' : 'is_holiday',
    'is_weekend' : 'is_weekend',
    'season' : 'season'
}

#rename the columns to the specified column names
bike.rename(new_cols_dict, axis = 1, inplace=True)  #Inplace = False --> show results

In [14]:
#create a season dictionary --> map the integer 0-3 to the actual written values according to Metadata
season_dict = {
    '0.0' : 'spring',
    '1.0' : 'summer',
    '2.0' : 'fall',
    '3.0' : 'winter'
}

#also create weather code dictionary according to Metadata
weather_dict = {
    '1.0':'Clear',
    '2.0':'Scattered clouds',
    '3.0':'Broken clouds',
    '4.0':'Cloudy',
    '7.0':'Rain',
    '10.0':'Rain with thunderstorm',
    '26.0':'Snowfall'
}

#changing the columns data type from integer to string --> to map new values
bike.season = bike.season.astype("str")
#mapping the values to the actual written season
bike.season = bike.season.map(season_dict)

#same for weather
bike.weather = bike.weather.astype("str")
bike.weather = bike.weather.map(weather_dict)

In [15]:
#Check dataset 
bike.head()

,time,count of a new bike shares,real temperature in C,temperature in C feels like,humidity_percent,wind_speed_km/h,weather,is_holiday,is_weekend,season
0,2015-01-04 00:00:00,182,3.0,2.0,93.0,6.0,Broken clouds,0.0,1.0,winter
1,2015-01-04 01:00:00,138,3.0,2.5,93.0,5.0,Clear,0.0,1.0,winter
2,2015-01-04 02:00:00,134,2.5,2.5,96.5,0.0,Clear,0.0,1.0,winter
3,2015-01-04 03:00:00,72,2.0,2.0,100.0,0.0,Clear,0.0,1.0,winter
4,2015-01-04 04:00:00,47,2.0,0.0,93.0,6.5,Clear,0.0,1.0,winter


In [16]:
# writing the final dataframe to an excel file that will be used in Tableau visualisations. 
# The file will be the 'london_bikes_final.xlsx' file and the sheet name is 'Data'
bike.to_excel('london_bikes_final.xlsx', sheet_name='Data')

In [17]:
#There is no sheet_name in csv file
bike.to_csv('london_bike.csv', )